# Run different TGLFs

In [ ]:
using Revise
using Printf
ENV["GACODE_ROOT_GPU"] = "/global/homes/t/tneiser/perl_gacode/gacode"
ENV["GACODE_ROOT_CPU"] = "/global/homes/t/tneiser/perl_gacode/gacode_cpu/gacode"
using TurbulentTransport
using TJLF

In [ ]:
# lets write an input.tglf to file

tmpdir = mktempdir()
filepath = joinpath(tmpdir, "input.tglf")
input_tglf_lines = """
ADIABATIC_ELEC = .false.
ALPHA_E = 1.0
ALPHA_MACH = 0.0
ALPHA_P = 1.0
ALPHA_QUENCH = 0
ALPHA_ZF = -1.0
AS_1 = 1.0
AS_2 = 0.929744
AS_3 = 0.0117093
BETAE = 0.00104401
BETA_LOC = 0.0
DAMP_PSI = 0.0
DAMP_SIG = 0.0
DEBYE = 0.0294130
DEBYE_FACTOR = 1.0
DELTA_LOC = 0.0646212
DRMAJDX_LOC = -0.100128
DRMINDX_LOC = 1.0
DZMAJDX_LOC = 0.00288598
ETG_FACTOR = 1.25
FILTER = 2.0
FIND_WIDTH = .true.
GCHAT = 1.0
GHAT = 1.0
GRADB_FACTOR = 0.0
IBRANCH = -1
IFLUX = .true.
KAPPA_LOC = 1.33468
KX0_LOC = 0.0
KY = 0.3
KYGRID_MODEL = 0
LINSKER_FACTOR = 0.0
MASS_1 = 0.000272444
MASS_2 = 1.0
MASS_3 = 6.0
NBASIS_MAX = 6
NBASIS_MIN = 2
NEW_EIKONAL = .true.
NKY = 16
NMODES = 2
NS = 3
NWIDTH = 21
NXGRID = 16
PARK = 1.0
P_PRIME_LOC = -0.000710683
Q_LOC = 1.63092
Q_PRIME_LOC = 10.0390
RLNP_CUTOFF = 18.0
RLNS_1 = 0.850818
RLNS_2 = 0.882123
RLNS_3 = 0.436538
RLTS_1 = 2.55493
RLTS_2 = 1.85863
RLTS_3 = 1.85863
RMAJ_LOC = 2.81750
RMIN_LOC = 0.585048
SAT_RULE = 0
SIGN_BT = -1
SIGN_IT = 1
S_DELTA_LOC = 0.124163
S_KAPPA_LOC = 0.134773
S_ZETA_LOC = -0.0252807
TAUS_1 = 1.0
TAUS_2 = 1.06073
TAUS_3 = 1.06073
THETA_TRAPPED = 0.7
UNITS = 'GYRO'
USE_AVE_ION_GRID = .false.
USE_BISECTION = .true.
USE_BPAR = .true.
USE_BPER = .true.
USE_INBOARD_DETRAPPED = .false.
USE_MHD_RULE = .false.
VEXB_SHEAR = 0.0572529
VPAR_1 = 0.265781
VPAR_2 = 0.265781
VPAR_3 = 0.265781
VPAR_MODEL = 0
VPAR_SHEAR_1 = 0.449679
VPAR_SHEAR_2 = 0.449679
VPAR_SHEAR_3 = 0.449679
VPAR_SHEAR_MODEL = 1
WDIA_TRAPPED = 1.0
WD_ZERO = 0.1
WIDTH = 1.65
WIDTH_MIN = 0.3
XNUE = 0.172756
XNU_FACTOR = 1.0
XNU_MODEL = 3
ZEFF = 1.35128
ZETA_LOC = -0.0105998
ZMAJ_LOC = 0.00353826
ZS_1 = -1.0
ZS_2 = 1.0
ZS_3 = 6.0
""";
open(filepath, "w") do f
    write(f, input_tglf_lines)
end;

In [ ]:
# lets load the input.tglf file
input_tglf = TurbulentTransport.load(InputTGLF(),filepath);
# Ensure TJLF runs with the same parameters in TGLF for USE_PRESETS=.true.
TurbulentTransport.apply_presets!(input_tglf)

In [ ]:
using ForwardDiff

# Build baseline InputTJLF{Float64}
input_tjlf = TJLF.InputTJLF{Float64}(input_tglf)

# Helper: construct InputTJLF{T} from a Float64 baseline
function convert_input_tjlf(::Type{T}, base::InputTJLF{Float64}) where {T<:Real}
    ns = base.NS
    nky = length(base.KY_SPECTRUM)
    inp = InputTJLF{T}(ns, nky)
    for fn in fieldnames(InputTJLF)
        v = getfield(base, fn)
        ismissing(v) && continue
        if v isa Float64
            setfield!(inp, fn, T(v))
        elseif v isa Vector{Float64}
            setfield!(inp, fn, T.(v))
        elseif v isa Vector{ComplexF64}
            setfield!(inp, fn, Complex{T}.(v))
        else
            setfield!(inp, fn, v)
        end
    end
    return inp
end

# Scalar function: RLTS_2 → Qe through full TJLF solver
function tjlf_Qe(x::T) where {T<:Real}
    inp = convert_input_tjlf(T, input_tjlf)
    inp.RLTS[2] = x
    QL = TJLF.run_tjlf(inp)
    return TJLF.Qe(QL)
end

x0 = input_tjlf.RLTS[2]
println("Testing ∂Qe/∂RLTS_2 at RLTS_2 = $x0\n")

# Finite differences (reference)
h = 1e-5
dQe_fd = (tjlf_Qe(x0 + h) - tjlf_Qe(x0 - h)) / (2h)
println("Finite differences: ∂Qe/∂RLTS_2 = $dQe_fd")

# ForwardDiff
print("ForwardDiff:        ")
try
    dQe_ad = ForwardDiff.derivative(tjlf_Qe, x0)
    relerr = abs(dQe_fd) > 0 ? abs(dQe_ad - dQe_fd) / abs(dQe_fd) : abs(dQe_ad - dQe_fd)
    println("∂Qe/∂RLTS_2 = $dQe_ad  (rel. error vs FD: $(round(relerr, sigdigits=3)))")
catch e
    println("FAILED — $(sprint(showerror, e))")
end

In [ ]:
# Run TGLF-NN model selector due to large number of available models
results = model_selector(
    input_tglf;
    filter_sat_rule=:sat3,
    electromagnetic=true,
    max_models=5, 
    ground_truth=true, 
    show_fluxes=false)

In [ ]:
# get a list of available models
#for model in TurbulentTransport.available_models()
#    println(model)
#end

In [ ]:
# run TGLF-NN
TurbulentTransport.run_tglfnn(input_tglf; model_filename="sat3_em_d3d_azf-1_withnegD", warn_nn_train_bounds=true)

In [ ]:
# compare with full TGLF
TurbulentTransport.run_tglf(input_tglf)

In [ ]:
# compare with full TJLF
TurbulentTransport.run_tjlf(input_tglf)

In [ ]:
# run GKNN
TurbulentTransport.run_tglfnn(input_tglf; model_filename="sat3_em_d3d_azf-1_withnegD", warn_nn_train_bounds=true, fidelity=:GKNN)

In [ ]:
# Run modular QLGYRO: linear CGYRO + TJLF saturation rules
state = TurbulentTransport.run_qlgyro(input_tglf;
    basedir        = "",            # defaults to /global/cfs/cdirs/m3739/results/FUSE/QLGYRO/qlgyro_<user>_<datetime>, fallback to $PSCRATCH
    kygrid_model   = 4,             # 4=TGLF default grid; 0=simple linear [0.1..1.2]
    ky_indices     = nothing,       # e.g. [1,2,3] to run a subset (1-indexed)
    gpu            = true,
    n_mpi          = 32,
    n_omp          = 4,
    walltime       = "00:15:00",
    repo           = "m3739_g",     # NERSC project allocation
    qos            = "regular",     # "regular", "premium", or "debug"
    wait_for_completion = false,
    poll_interval  = 60,            # seconds between convergence checks (when waiting_for_completion=true)
)

In [ ]:
# Check status / refresh (re-run this cell to monitor progress)
# Once all runs converge, compute fluxes
# state = TurbulentTransport.load_run_state("/global/cfs/cdirs/m3739/results/FUSE/QLGYRO/qlgyro_tneiser_20260409_115602")
state = TurbulentTransport.load_run_state(state.basedir)
TurbulentTransport.refresh_qlgyro!(state;
    qos            = "regular",
    repo           = "m3739_g",
    gpu            = true,
    n_mpi          = 32,
    n_omp          = 4,
    walltime       = "00:15:00",
    max_resubmits  = 3,
)

In [ ]:
# Compute fluxes once converged (or run the full workflow with wait)
converged_indices = findall(state.converged)
println("Converged ky indices: ", converged_indices, " ($(length(converged_indices))/$(length(state.converged)))")

qlgyro_result = TurbulentTransport.compute_qlgyro_fluxes(input_tglf, state)
println("Modular QLGYRO result:")
display(qlgyro_result)

# Compare with QLGYRO-NN
QLGYRO_NN_result = TurbulentTransport.run_tglfnn(input_tglf; model_filename="sat3_em_d3d_azf-1_withnegD", warn_nn_train_bounds=false, fidelity=:GKNN)
println("\nQLGYRO-NN result (for reference):")
display(QLGYRO_NN_result)

In [ ]:
# Compare QLGYRO with TJLF for all available saturation rules and alpha_zf values
f3(x) = @sprintf("%.3g", x)
lines = String[]
for (sat, azf) in [(1, 1.0), (1, -1.0), (2, 1.0), (2, -1.0), (3, 1.0), (3, -1.0)]
    header = "SAT_RULE=$sat, ALPHA_ZF=$azf:"
    println(header)
    push!(lines, header)
    
    q = TurbulentTransport.compute_qlgyro_fluxes(input_tglf, state; sat_rule=sat, alpha_zf=azf)
    qlgyro_line = "  QLGYRO:  Qe=$(f3(q.ENERGY_FLUX_e))  Qi=$(f3(q.ENERGY_FLUX_i))  Γe=$(f3(q.PARTICLE_FLUX_e))  Γi=[$(join(f3.(q.PARTICLE_FLUX_i), ", "))]  Πi=$(f3(q.STRESS_TOR_i))"
    println(qlgyro_line)
    push!(lines, qlgyro_line)
    
    input_tglf_override = deepcopy(input_tglf)
    input_tglf_override.SAT_RULE = sat
    input_tglf_override.ALPHA_ZF = azf
    TurbulentTransport.apply_presets!(input_tglf_override)
    if sat == 1
        input_tglf_override.UNITS = "GYRO"
    end
    t = TurbulentTransport.run_tjlf(input_tglf_override)
    tjlf_line = "  TJLF:    Qe=$(f3(t.ENERGY_FLUX_e))  Qi=$(f3(t.ENERGY_FLUX_i))  Γe=$(f3(t.PARTICLE_FLUX_e))  Γi=[$(join(f3.(t.PARTICLE_FLUX_i), ", "))]  Πi=$(f3(t.STRESS_TOR_i))"
    println(tjlf_line)
    println()
    push!(lines, tjlf_line)
    push!(lines, "")
end

In [ ]:
# 4-panel comparison: CGYRO linear vs TJLF linear
using Plots, LaTeXStrings
gr()
default(fontfamily="Computer Modern")

# --- CGYRO data from QLGYRO run state ---
conv = state.converged
nky_total = length(state.ky_values)
conv_indices = [i for i in 1:nky_total if conv[i]]
ky_cgyro = state.ky_values[conv]
nky_c = length(ky_cgyro)

# parse_cgyro_eigenvalue returns (freq::Float64, gamma::Float64) — scalars, one mode
gamma_c = zeros(nky_c)
freq_c  = zeros(nky_c)
for (j, iky) in enumerate(conv_indices)
    ky_dir = joinpath(state.basedir, "KY_$iky")
    f, g = TurbulentTransport.parse_cgyro_eigenvalue(ky_dir)
    gamma_c[j] = g
    freq_c[j]  = f
end

# --- TJLF data: run on InputTJLF so we keep KY_SPECTRUM ---
import TJLF
input_tglf_tjlf = deepcopy(input_tglf)
input_tglf_tjlf.SAT_RULE = 3
input_tglf_tjlf.ALPHA_ZF = -1.0
input_tjlf = TJLF.InputTJLF{Float64}(input_tglf_tjlf)
result = TJLF.run(input_tjlf)  # populates input_tjlf.KY_SPECTRUM in-place
ky_tjlf = input_tjlf.KY_SPECTRUM
# eigenvalue: (nmodes, nky, 2) where [:,:,1]=gamma, [:,:,2]=freq
gamma_t1 = result.eigenvalue[1, :, 1]  # mode 1, (nky,)
freq_t1  = result.eigenvalue[1, :, 2]
gamma_t2 = result.eigenvalue[2, :, 1]  # mode 2, (nky,)
freq_t2  = result.eigenvalue[2, :, 2]

# QL energy weights from TJLF: QL_weights is (3_fields, ns, nmodes, nky, 5_types)
ql_weights_t = result.QL_weights

# --- CGYRO QL flux (for QL weight comparison) ---
ns = input_tglf.NS
nfield = (coalesce(input_tglf.USE_BPER, false) ? 1 : 0) + (coalesce(input_tglf.USE_BPAR, false) ? 1 : 0) + 1
ql_cgyro_all = zeros(ns, nfield, 3, nky_c)  # (ns, nfield, 3moments, nky)
for (j, iky) in enumerate(conv_indices)
    ky_dir = joinpath(state.basedir, "KY_$iky")
    ql = TurbulentTransport.parse_cgyro_qlflux(ky_dir, ns, nfield)
    ql_cgyro_all[:, :, :, j] .= Float64.(ql) .* ky_cgyro[j]
end
# Reorder CGYRO species to TJLF convention (electrons first)
ql_cgyro_reordered = similar(ql_cgyro_all)
ql_cgyro_reordered[1, :, :, :] .= ql_cgyro_all[end, :, :, :]  # electrons
for s in 1:ns-1
    ql_cgyro_reordered[s+1, :, :, :] .= ql_cgyro_all[s, :, :, :]
end

# --- Plotting (both modes) ---
# Panel 1: γ vs ky
p1 = plot(xlabel=L"k_y \rho_s", ylabel=L"\gamma \, [c_s/a]", title="Growth Rate",
          legend=:topleft, xscale=:log10, yscale=:log10, ylims=(1e-3, 10))
plot!(p1, ky_cgyro, gamma_c, marker=:circle, ms=4, lw=0, color=:black, label="CGYRO")
plot!(p1, ky_tjlf, gamma_t1, lw=2, color=:blue, label="TGLF")

# Panel 2: ω vs ky (symlog via asinh transform)
linthresh = 0.1  # linear region around zero
symlog(x) = sign(x) * log10(1 + abs(x) / linthresh)
symlog_inv(y) = sign(y) * linthresh * (10^abs(y) - 1)
# Custom tick values
tick_vals_raw = [-10, -1, -0.1, 0, 0.1, 1, 10]
tick_vals_t = symlog.(tick_vals_raw)
tick_labels = string.(tick_vals_raw)

p2 = plot(xlabel=L"k_y \rho_s", ylabel=L"\omega \, [c_s/a]", title="Frequency",
          legend=:bottomleft, xscale=:log10, yticks=(tick_vals_t, tick_labels))
plot!(p2, ky_cgyro, symlog.(freq_c), marker=:circle, ms=4, lw=0, color=:black, label="CGYRO")
plot!(p2, ky_tjlf, symlog.(-freq_t1), lw=2, color=:blue, label="TGLF")

# Panel 3: γ/ky vs ky (mixing length estimate)
p3 = plot(xlabel=L"k_y \rho_s", ylabel=L"\gamma / k_y", title="Mixing Rate",
          legend=:topright, xscale=:log10, yscale=:log10, ylims=(1e-3, 10))
plot!(p3, ky_cgyro, gamma_c ./ ky_cgyro, marker=:circle, ms=4, lw=0, color=:black, label="CGYRO")
plot!(p3, ky_tjlf, gamma_t1 ./ ky_tjlf, lw=2, color=:blue, label="TGLF")
# ALPHA_ZF=-1 low-ky cutoff: same logic as get_zonal_mixing in TJLF
let satParams_t = TJLF.get_sat_params(input_tjlf)
    rho_ion = 0.0; charge = 0.0
    for is in 2:input_tjlf.NS
        if !input_tjlf.USE_AVE_ION_GRID
            rho_ion = sqrt(input_tjlf.TAUS[2]*input_tjlf.MASS[2]) / abs(input_tjlf.ZS[2]); break
        elseif input_tjlf.ZS[is]*input_tjlf.AS[is]/abs(input_tjlf.ZS[1]*input_tjlf.AS[1]) > 0.1
            charge += input_tjlf.ZS[is]*input_tjlf.AS[is]
            rho_ion += input_tjlf.AS[is]*sqrt(input_tjlf.TAUS[is]*input_tjlf.MASS[is])
        end
    end
    if charge != 0.0; rho_ion = rho_ion/charge; end
    kymin_azf = 0.173 * sqrt(2.0) / rho_ion
    if input_tjlf.SAT_RULE == 2 || input_tjlf.SAT_RULE == 3
        kymin_azf *= satParams_t.grad_r0
    end
    vline!(p3, [kymin_azf], color=:red, linestyle=:dash, lw=1.5, label="ALPHA_ZF=-1 cutoff")
end

# Panel 4: QL energy weights (summed over fields)
# TGLF ion energy: species 2, energy type (index 2)
ql_energy_tjlf1 = vec(dropdims(sum(ql_weights_t[:, 2, 1:1, :, 2], dims=1), dims=1))
# TGLF electron energy: species 1, energy type (index 2)
ql_energy_e_tjlf1 = vec(dropdims(sum(ql_weights_t[:, 1, 1:1, :, 2], dims=1), dims=1))
# CGYRO ion: reordered species 2, moment index 2 = energy, sum over fields
ql_energy_cgyro = dropdims(sum(ql_cgyro_reordered[2:2, :, 2:2, :], dims=(1,2,3)), dims=(1,2,3))  # (nky,)
# CGYRO electron: reordered species 1, moment index 2 = energy, sum over fields
ql_energy_e_cgyro = dropdims(sum(ql_cgyro_reordered[1:1, :, 2:2, :], dims=(1,2,3)), dims=(1,2,3))

# filter zeros for log scale, use abs values
ql_ci = abs.(ql_energy_cgyro);   m_ci = ql_ci .> 0
ql_ce = abs.(ql_energy_e_cgyro); m_ce = ql_ce .> 0
ql_ti = abs.(ql_energy_tjlf1);   m_ti = ql_ti .> 0
ql_te = abs.(ql_energy_e_tjlf1); m_te = ql_te .> 0
ql4_all = [ql_ci[m_ci]; ql_ce[m_ce]; ql_ti[m_ti]; ql_te[m_te]]
ql4_ymax = isempty(ql4_all) ? 1.0 : 1.1 * maximum(ql4_all)
ql4_ymin = isempty(ql4_all) ? 0.01 : 0.5 * minimum(ql4_all)

p4 = plot(xlabel=L"k_y \rho_s", ylabel="QL energy weight", title="QL Weights (energy)",
          legend=:topleft, xscale=:log10, yscale=:log10, ylims=(ql4_ymin, ql4_ymax))
plot!(p4, ky_cgyro[m_ci], ql_ci[m_ci], marker=:circle, ms=4, lw=0, color=:black, label="CGYRO ion")
plot!(p4, ky_cgyro[m_ce], ql_ce[m_ce], marker=:diamond, ms=4, lw=0, color=:gray, label="CGYRO elec")
plot!(p4, ky_tjlf[m_ti], ql_ti[m_ti], lw=2, color=:blue, label="TGLF ion")
plot!(p4, ky_tjlf[m_te], ql_te[m_te], lw=2, color=:cyan, label="TGLF elec")

fig = plot(p1, p2, p3, p4, layout=(2,2), size=(1000, 700), margin=5Plots.mm)
savefig(fig, joinpath(@__DIR__, "cgyro_vs_tglf_linear.pdf"))
fig


In [ ]:
# Saturated intensity spectrum (dimensionless field intensity, before QL weighting)
# intensity_factor shape: (nky, nmodes) — pure saturation rule output
# SAT1/2/3 x AZF=-1/+1: solid lines for AZF=-1, dashed for AZF=+1
using Plots, LaTeXStrings, TJLF
gr()
default(fontfamily="Computer Modern")

configs    = [(1, -1.0, "SAT1 AZF=-1"), (1,  1.0, "SAT1 AZF=+1"),
              (2, -1.0, "SAT2 AZF=-1"), (2,  1.0, "SAT2 AZF=+1"),
              (3, -1.0, "SAT3 AZF=-1"), (3,  1.0, "SAT3 AZF=+1")]
colors_cfg = [:navy, :cornflowerblue, :darkred, :salmon, :darkgreen, :mediumseagreen]
styles_cfg = [:solid, :dash, :solid, :dash, :solid, :dash]
marker_cfg = [:circle, :circle, :diamond, :diamond, :utriangle, :utriangle]

ky_ticks = ([0.05, 0.1, 0.2, 0.5, 1.0, 2.0], ["0.05", "0.1", "0.2", "0.5", "1", "2"])

p_int = plot(xlabel=L"k_y \rho_s", ylabel="Intensity [a.u.]",
             title="Saturated Intensity Spectrum",
             legend=:bottomleft, xscale=:log10, yscale=:log10,
             xlims=(0.04, 3.0), xticks=ky_ticks)

# Compute kymin for ALPHA_ZF=-1 cutoff (same logic as get_zonal_mixing in TJLF)
kymin_azf_sat1, kymin_azf_sat23 = let
    inp_tmp = TJLF.InputTJLF{Float64}(let x = deepcopy(input_tglf); x.SAT_RULE = 3; x.ALPHA_ZF = -1.0; x end)
    satP = TJLF.get_sat_params(inp_tmp)
    rho_ion = 0.0; charge = 0.0
    for is in 2:inp_tmp.NS
        if !inp_tmp.USE_AVE_ION_GRID
            rho_ion = sqrt(inp_tmp.TAUS[2]*inp_tmp.MASS[2]) / abs(inp_tmp.ZS[2]); break
        elseif inp_tmp.ZS[is]*inp_tmp.AS[is]/abs(inp_tmp.ZS[1]*inp_tmp.AS[1]) > 0.1
            charge += inp_tmp.ZS[is]*inp_tmp.AS[is]
            rho_ion += inp_tmp.AS[is]*sqrt(inp_tmp.TAUS[is]*inp_tmp.MASS[is])
        end
    end
    if charge != 0.0; rho_ion = rho_ion/charge; end
    kymin_base = 0.173 * sqrt(2.0) / rho_ion
    kymin_base, kymin_base * satP.grad_r0
end

all_intensities = Float64[]

for (cfg_idx, (sat, azf, lbl)) in enumerate(configs)
    local inp_tglf = let x = deepcopy(input_tglf); x.SAT_RULE = sat; x.ALPHA_ZF = azf; x end
    local inp  = TJLF.InputTJLF{Float64}(inp_tglf)
    local res  = TJLF.run(inp)
    local ky   = inp.KY_SPECTRUM
    local satP = TJLF.get_sat_params(inp)
    local gamma_matrix = res.eigenvalue[:, :, 1]  # (nmodes, nky)
    local QL_weights   = res.QL_weights

    # intensity_sat needs zonal mixing params for SAT2/3
    local intensity_factor
    if sat == 2 || sat == 3
        vzf_out, kymax_out, jmax_out = TJLF.get_zonal_mixing(inp, satP, gamma_matrix[1, :])
        intensity_factor, = TJLF.intensity_sat(inp, satP, gamma_matrix, QL_weights;
                                               vzf_out_param=vzf_out,
                                               kymax_out_param=kymax_out,
                                               jmax_out_param=jmax_out)
    else
        intensity_factor, = TJLF.intensity_sat(inp, satP, gamma_matrix, QL_weights)
    end
    # intensity_factor: (nky, nmodes) — filter non-positive values for log scale
    intensity_mode1 = intensity_factor[:, 1]
    mask = intensity_mode1 .> 0
    append!(all_intensities, intensity_mode1[mask])
    plot!(p_int, ky[mask], intensity_mode1[mask], lw=2,
          color=colors_cfg[cfg_idx], linestyle=styles_cfg[cfg_idx],
          marker=marker_cfg[cfg_idx], ms=5, markerstrokewidth=0,
          label=lbl)
end

# ALPHA_ZF=-1 low-ky cutoff lines
vline!(p_int, [kymin_azf_sat23], color=:black, linestyle=:dot, lw=1.5, label="kymin AZF=-1 (SAT2/3)")
vline!(p_int, [kymin_azf_sat1],  color=:gray,  linestyle=:dot, lw=1.5, label="kymin AZF=-1 (SAT1)")

fig2 = plot(p_int, size=(700, 500), margin=5Plots.mm)
savefig(fig2, joinpath(@__DIR__, "intensity_spectrum.pdf"))
fig2


In [ ]:
# QLGYRO vs TGLF: Intensity, QL Weight, Electron Heat Flux Spectrum
using Plots, LaTeXStrings, TJLF
gr()
default(fontfamily="Computer Modern")

# === User-defined parameters ===
SAT = 3
AZF = -1.0
SPECIES = :electron  # :electron or :ion

# ──────────────────────── TGLF ────────────────────────
inp_tglf_t = let x = deepcopy(input_tglf); x.SAT_RULE = SAT; x.ALPHA_ZF = AZF; x end
inp_tjlf_t = TJLF.InputTJLF{Float64}(inp_tglf_t)
res_t = TJLF.run(inp_tjlf_t)
ky_t = copy(inp_tjlf_t.KY_SPECTRUM)
satP_t = TJLF.get_sat_params(inp_tjlf_t)
gamma_mat_t = res_t.eigenvalue[:, :, 1]   # (nmodes, nky)
ql_w_t = res_t.QL_weights                 # (nfield, ns, nmodes, nky, 5)

int_t, _, QLA_E_t, _ = if SAT in (2, 3)
    vzf, km, jm = TJLF.get_zonal_mixing(inp_tjlf_t, satP_t, gamma_mat_t[1, :])
    TJLF.intensity_sat(inp_tjlf_t, satP_t, gamma_mat_t, ql_w_t;
        vzf_out_param=vzf, kymax_out_param=km, jmax_out_param=jm)
else
    TJLF.intensity_sat(inp_tjlf_t, satP_t, gamma_mat_t, ql_w_t)
end

# QL energy weight (sum over fields): mode 1, type 2
sp_idx_t = SPECIES == :electron ? 1 : 2
ql_E_t = vec(sum(ql_w_t[:, sp_idx_t, 1, :, 2], dims=1))   # (nky_t,)
flux_E_t = int_t[:, 1] .* QLA_E_t[1] .* ql_E_t

# ──────────────────────── QLGYRO ────────────────────────
# Mirror compute_qlgyro_fluxes exactly: use ALL ky values, gamma=0 for unconverged
nky_all = length(state.ky_values)
ns_loc  = input_tglf.NS
nf_loc  = (coalesce(input_tglf.USE_BPER, false) ? 1 : 0) +
          (coalesce(input_tglf.USE_BPAR, false) ? 1 : 0) + 1
nmodes_c = 1

# CGYRO eigenvalues (all ky, gamma=0 for unconverged)
gamma_cgyro = zeros(Float64, nky_all)
for iky in 1:nky_all
    if state.converged[iky]
        ky_dir = joinpath(state.basedir, "KY_$iky")
        _, g = TurbulentTransport.parse_cgyro_eigenvalue(ky_dir)
        gamma_cgyro[iky] = max(g, 0.0)
    end
end

# CGYRO QL flux → TJLF-convention QL_weights
ql_flux_cgyro = zeros(Float64, nky_all, ns_loc, nf_loc, 3)
for iky in 1:nky_all
    if state.converged[iky]
        ky_dir = joinpath(state.basedir, "KY_$iky")
        ql_raw = TurbulentTransport.parse_cgyro_qlflux(ky_dir, ns_loc, nf_loc)
        ql_flux_cgyro[iky, :, :, :] .= Float64.(ql_raw) .* state.ky_values[iky]
    end
end
# Species reorder: CGYRO [ion,..,e] → TJLF [e, ion,..]
ql_flux_tjlf = similar(ql_flux_cgyro)
ql_flux_tjlf[:, 1, :, :] .= ql_flux_cgyro[:, end, :, :]
for s in 1:ns_loc-1
    ql_flux_tjlf[:, s+1, :, :] .= ql_flux_cgyro[:, s, :, :]
end
QL_w_c = zeros(Float64, nf_loc, ns_loc, nmodes_c, nky_all, 5)
for iky in 1:nky_all, is in 1:ns_loc, ifl in 1:nf_loc
    QL_w_c[ifl, is, 1, iky, 1] = ql_flux_tjlf[iky, is, ifl, 1]
    QL_w_c[ifl, is, 1, iky, 2] = ql_flux_tjlf[iky, is, ifl, 2]
    QL_w_c[ifl, is, 1, iky, 3] = ql_flux_tjlf[iky, is, ifl, 3]
end

# Build InputTJLF matching reference: 2-arg constructor + update_input_tjlf!
inp_tglf_c = deepcopy(input_tglf)
inp_tglf_c.SAT_RULE = SAT
inp_tglf_c.ALPHA_ZF = AZF
inp_tjlf_c = TJLF.InputTJLF{Float64}(ns_loc, nky_all)
TJLF.update_input_tjlf!(inp_tjlf_c, inp_tglf_c)
inp_tjlf_c.SAT_RULE = SAT
inp_tjlf_c.ALPHA_ZF = Float64(AZF)
inp_tjlf_c.KYGRID_MODEL = 0
inp_tjlf_c.NKY = nky_all
inp_tjlf_c.NMODES = nmodes_c
inp_tjlf_c.KY_SPECTRUM .= state.ky_values
inp_tjlf_c.WIDTH_SPECTRUM .= inp_tjlf_c.WIDTH

gamma_mat_c = zeros(Float64, nmodes_c, nky_all)
gamma_mat_c[1, :] .= gamma_cgyro
satP_c = TJLF.get_sat_params(inp_tjlf_c)

int_c, _, QLA_E_c, _ = if SAT in (2, 3)
    vzf_c, km_c, jm_c = TJLF.get_zonal_mixing(inp_tjlf_c, satP_c, gamma_cgyro)
    TJLF.intensity_sat(inp_tjlf_c, satP_c, gamma_mat_c, QL_w_c;
        vzf_out_param=vzf_c, kymax_out_param=km_c, jmax_out_param=jm_c)
else
    TJLF.intensity_sat(inp_tjlf_c, satP_c, gamma_mat_c, QL_w_c)
end

ky_c = state.ky_values
sp_idx_c = SPECIES == :electron ? 1 : 2
ql_E_c = vec(sum(QL_w_c[:, sp_idx_c, 1, :, 2], dims=1))
flux_E_c = int_c[:, 1] .* QLA_E_c[1] .* ql_E_c

# ─── Diagnostics: verify data is different ───
println("=== Diagnostic: TGLF vs QLGYRO ===")
println("ky grids same? ", ky_t ≈ ky_c)
println("nky: TGLF=$(length(ky_t)), QLGYRO=$(length(ky_c))")
println("Growth rates (first 5):")
println("  TGLF:   ", round.(gamma_mat_t[1, 1:min(5,end)], sigdigits=5))
println("  CGYRO:  ", round.(gamma_cgyro[1:min(5,end)], sigdigits=5))
println("Intensity (first 5):")
println("  TGLF:   ", round.(int_t[1:min(5,end), 1], sigdigits=5))
println("  QLGYRO: ", round.(int_c[1:min(5,end), 1], sigdigits=5))
println("QLA_E:  TGLF=$(round(QLA_E_t[1], sigdigits=5)), QLGYRO=$(round(QLA_E_c[1], sigdigits=5))")

# ──────────────────────── Plotting ────────────────────────
ky_ticks = ([0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0],
            ["0.05", "0.1", "0.2", "0.5", "1", "2", "5"])

# Panel 0: Growth rate comparison (linear y)
p0 = plot(xlabel=L"k_y \rho_s", ylabel=L"\gamma / k_y ",
          title=L"\gamma / k_y", legend=:topright,
          xscale=:log10, xlim=(0.05, 5), xticks=ky_ticks)
plot!(p0, ky_c, gamma_cgyro ./ ky_c, marker=:circle, ms=4, lw=0,
      color=:black, label="CGYRO")
plot!(p0, ky_t, gamma_mat_t[1, :] ./ ky_t, lw=2, color=:blue, label="TGLF")
# ALPHA_ZF=-1 low-ky cutoff
let satP0 = TJLF.get_sat_params(inp_tjlf_t)
    rho_ion = 0.0; charge = 0.0
    for is in 2:inp_tjlf_t.NS
        if !inp_tjlf_t.USE_AVE_ION_GRID
            rho_ion = sqrt(inp_tjlf_t.TAUS[2]*inp_tjlf_t.MASS[2]) / abs(inp_tjlf_t.ZS[2]); break
        elseif inp_tjlf_t.ZS[is]*inp_tjlf_t.AS[is]/abs(inp_tjlf_t.ZS[1]*inp_tjlf_t.AS[1]) > 0.1
            charge += inp_tjlf_t.ZS[is]*inp_tjlf_t.AS[is]
            rho_ion += inp_tjlf_t.AS[is]*sqrt(inp_tjlf_t.TAUS[is]*inp_tjlf_t.MASS[is])
        end
    end
    if charge != 0.0; rho_ion = rho_ion/charge; end
    kymin_azf = 0.173 * sqrt(2.0) / rho_ion
    if SAT in (2, 3); kymin_azf *= satP0.grad_r0; end
    vline!(p0, [kymin_azf], color=:red, linestyle=:dash, lw=1.5, label="AZF=-1 cutoff")
end

# Panel 1: Intensity
mt = int_t[:, 1] .> 0;  mc = int_c[:, 1] .> 0
p1 = plot(xlabel=L"k_y \rho_s", ylabel="Intensity [a.u.]",
          title="Saturated Intensity", legend=:topright,
          xscale=:log10, yscale=:linear, xlim=(0.05, 5), xticks=ky_ticks)
plot!(p1, ky_c[mc], int_c[mc, 1], marker=:circle, ms=4, lw=0,
      color=:black, label="QLGYRO")
plot!(p1, ky_t[mt], int_t[mt, 1], lw=2, color=:blue, label="TGLF")

# Panel 2: QL energy weight (log scale, filter non-positive)
mwt = ql_E_t .> 0
mwc = ql_E_c .> 0
ql_ymax = let vals = [ql_E_c[mwc]; ql_E_t[mwt]]
    isempty(vals) ? 1.0 : 1.1 * maximum(vals)
end
p2 = plot(xlabel=L"k_y \rho_s", ylabel="QL weight",
          title="QL Weight ($(SPECIES == :electron ? L"Q_e" : L"Q_i"))", legend=:topright,
          xscale=:log10, yscale=:log10, xlim=(0.05, 5), ylims=(:auto, ql_ymax), xticks=ky_ticks)
plot!(p2, ky_c[mwc], ql_E_c[mwc], marker=:circle, ms=4, lw=0,
      color=:black, label="QLGYRO")
plot!(p2, ky_t[mwt], ql_E_t[mwt], lw=2, color=:blue, label="TGLF")

# Panel 3: Electron heat flux spectrum
mft = flux_E_t .!= 0;  mfc = flux_E_c .!= 0
p3 = plot(xlabel=L"k_y \rho_s", ylabel="\$Q_$(SPECIES == :electron ? "e" : "i")\$ spectrum",
          title="$(SPECIES == :electron ? "Electron" : "Ion") Heat Flux Spectrum", legend=:topright,
          xscale=:log10, xlim=(0.05, 5), xticks=ky_ticks)
plot!(p3, ky_c[mfc], abs.(flux_E_c[mfc]), marker=:circle, ms=4, lw=0,
      color=:black, label="QLGYRO")
plot!(p3, ky_t[mft], abs.(flux_E_t[mft]), lw=2, color=:blue, label="TGLF")

fig3 = plot(p0, p1, p2, p3, layout=(2, 2), size=(1000, 700), margin=5Plots.mm,
            plot_title="SAT$SAT, ALPHA_ZF=$AZF")
savefig(fig3, joinpath(@__DIR__, "qlgyro_vs_tglf_spectrum.pdf"))
fig3


In [ ]:
# Sumflux spectrum: ky-resolved flux contribution for TGLF vs QLGYRO
# Depends on cell 16 variables: inp_tjlf_t, res_t, inp_tjlf_c, gamma_mat_c, QL_w_c, satP_c, SAT, AZF, SPECIES
using Plots, LaTeXStrings, TJLF
gr()
default(fontfamily="Computer Modern")

# --- TGLF: flux_spectrum already in res_t ---
flux_spec_t = res_t.flux_spectrum   # (nfield, ns, nmodes, nky_t, 5)

# --- QLGYRO: compute via sum_ky_spectrum ---
if SAT in (2, 3)
    vzf_c2, km_c2, jm_c2 = TJLF.get_zonal_mixing(inp_tjlf_c, satP_c, gamma_mat_c[1, :])
    _, flux_spec_c = TJLF.sum_ky_spectrum(inp_tjlf_c, satP_c, gamma_mat_c, QL_w_c;
        vzf_out_param=vzf_c2, kymax_out_param=km_c2, jmax_out_param=jm_c2)
else
    _, flux_spec_c = TJLF.sum_ky_spectrum(inp_tjlf_c, satP_c, gamma_mat_c, QL_w_c)
end

# --- Extract per-ky flux: sum over fields and modes ---
sp_idx = SPECIES == :electron ? 1 : 2
sp_lbl = SPECIES == :electron ? "e" : "i"

Qky_t_full = vec(sum(flux_spec_t[:, sp_idx:sp_idx, :, :, 2], dims=(1,3)))
Qky_c_full = vec(sum(flux_spec_c[:, sp_idx:sp_idx, :, :, 2], dims=(1,3)))

ky_t_s = inp_tjlf_t.KY_SPECTRUM
ky_c_s = inp_tjlf_c.KY_SPECTRUM
mask_t = ky_t_s .<= 5.0
mask_c = ky_c_s .<= 5.0
idx_t = collect(1:sum(mask_t))
idx_c = collect(1:sum(mask_c))
ky_labels_t = [i % 3 == 1 ? string(round(ky_t_s[mask_t][i], sigdigits=2)) : "" for i in idx_t]

pflux = plot(ylabel=latexstring("Q_{$(sp_lbl)} \\; [\\mathrm{gB}]"),
             title="$(SPECIES == :electron ? "Electron" : "Ion") Heat Flux Spectrum  (SAT$(SAT), AZF=$(AZF))",
             xlabel=L"k_y \rho_s",
             legend=:topright,
             xticks=(idx_t, ky_labels_t))

bar!(pflux, idx_t, Qky_t_full[mask_t]; bar_width=0.4, alpha=0.5, color=:blue,  label="TGLF")
bar!(pflux, idx_c, Qky_c_full[mask_c]; bar_width=0.4, alpha=0.5, color=:black, label="QLGYRO")

fig4 = plot(pflux, size=(700, 450), margin=5Plots.mm)
savefig(fig4, joinpath(@__DIR__, "sumflux_spectrum.pdf"))
fig4